# Étape 1 — Collecte des posts déclarés en français

Objectif : écouter le flux public Jetstream et stocker les publications déclarées `fr` dans SQLite. Aucun filtrage d'actualité, analyse thématique ou encodage n'est effectué.

Ce notebook est l'entrée interactive de la collecte. Les fonctions sont définies et expliquées dans `00_collecteur_jetstream.ipynb`. La collecte réseau est désactivée par défaut pour pouvoir lire et exécuter le notebook sans ajouter de données.

## 1. Chemins et imports

In [1]:
from pathlib import Path
import sys

# Fonctionne depuis le notebook, le sous-projet ou la racine du projet.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le projet Bluesky ou son dossier notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATABASE_PATH = ROOT / "posts.db"
print("Sous-projet :", ROOT.name)
print("Base SQLite :", (Path(ROOT.name) / DATABASE_PATH.relative_to(ROOT)).as_posix())

Sous-projet : bluesky-news-mining
Base SQLite : bluesky-news-mining/posts.db


In [2]:
from contextlib import closing
import json
import sqlite3

# Charger uniquement les cellules de fonctions, sans lancer de collecte.
collector_path = ROOT / "notebooks" / "00_collecteur_jetstream.ipynb"
collector_notebook = json.loads(collector_path.read_text(encoding="utf-8"))
for cell in collector_notebook["cells"]:
    if cell["cell_type"] == "code" and "collector-definitions" in cell["metadata"].get("tags", []):
        exec(compile("".join(cell["source"]), str(collector_path), "exec"), globals())

print("Flux :", URL)


Flux : wss://jetstream2.us-east.bsky.network/subscribe?wantedCollections=app.bsky.feed.post


## 2. Paramètres de collecte

`MAX_POSTS` compte les nouvelles URI françaises enregistrées pendant cette exécution, et non le nombre total de lignes de la base. `0` désactive le plafond. La durée est un second critère d'arrêt : le premier seuil atteint arrête la collecte. Pour demander 1 000 nouvelles publications sans limite de durée, définir `MAX_POSTS = 1000` et `DURATION_MINUTES = None`.

La base existante est conservée ; ce notebook ne la vide pas. Pour éviter de lancer la collecte en exécutant toutes les cellules, laisser `RUN_COLLECTION = False` tant que l'on souhaite seulement consulter les données.

In [3]:
RUN_COLLECTION = False
MAX_POSTS = 1000
DURATION_MINUTES = None

if MAX_POSTS < 0:
    raise ValueError("MAX_POSTS doit être positif ou nul.")
if DURATION_MINUTES is not None and DURATION_MINUTES <= 0:
    raise ValueError("La durée doit être strictement positive.")
print("Collecte activée :", RUN_COLLECTION)
print("Nouvelles URI à collecter :", MAX_POSTS or "sans plafond")

Collecte activée : False
Nouvelles URI à collecter : 1000


## 3. Filtrage et stockage

Pour chaque événement de la collection `app.bsky.feed.post`, le collecteur :

1. Construit l'URI unique du post à partir de l'auteur, de la collection et de la clé du record.
2. Retire de la base le post si un événement de suppression est reçu.
3. Pour une création ou mise à jour, ne conserve le post que si sa liste `langs` contient exactement `fr` (les posts multilingues sont acceptés).
4. Enregistre le texte original, l'auteur, la date, les langues et les indicateurs de réponse et de lien.

Le filtre utilise une langue déclarée, pas une détection linguistique. Les posts sans langue déclarée sont ignorés. Une URI existante est mise à jour plutôt que dupliquée.

## 4. Lancement de la collecte

Jupyter dispose déjà d'une boucle asynchrone : on appelle directement `await collect_posts(...)`, sans `asyncio.run`. Pour collecter, définir `RUN_COLLECTION = True` dans la cellule de paramètres, puis exécuter cette cellule. Le noyau doit avoir accès au réseau. Interrompre la cellule arrête la collecte.

Le collecteur tente de se reconnecter après une coupure avec un curseur en mémoire. Ce curseur n'est pas sauvegardé après l'arrêt du processus ; la reprise sans lacune n'est pas garantie.

In [4]:
if RUN_COLLECTION:
    await collect_posts(
        max_posts=MAX_POSTS,
        duration_minutes=DURATION_MINUTES,
        database_path=DATABASE_PATH,
    )
else:
    print("Collecte non lancée. Mettre RUN_COLLECTION = True pour l'activer.")

Collecte non lancée. Mettre RUN_COLLECTION = True pour l'activer.


## 5. Vérifier le nombre de posts stockés

La lecture est seule autorisée dans cette cellule : aucune base vide n'est créée par accident. Une différence entre le compteur de collecte et les lignes stockées peut notamment provenir des suppressions reçues.

In [5]:
if DATABASE_PATH.exists():
    with closing(sqlite3.connect(DATABASE_PATH.as_uri() + "?mode=ro", uri=True)) as connection:
        stored = connection.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
    print("Publications actuellement stockées :", stored)
else:
    print("Aucune base pour le moment : lancer d'abord une collecte.")

Publications actuellement stockées : 193
